In [8]:
from pwn import *

In [ ]:
HOST = "34.47.176.25"
PORT = 4901

def test(io):
    io.recvuntil(b"> ")
    io.sendline(b"2")

    io.recvuntil(b"Transmission: ")
    io.sendline(b"AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA")

    io.recvuntil(b"Ciphertext:\n")
    ct = io.recvline().strip().decode()
    return bytes.fromhex(ct)

io = remote(HOST, PORT)

ct = test(io)
blocks = [ct[i:i+16] for i in range(0,len(ct),16)]
print(f"ciphertext: {ct}")
print(f"blocks    : {blocks}")
if len(blocks) != len(set(blocks)):
    print("ebc, good...")
io.close()

[x] Opening connection to 34.47.176.25 on port 4901
[x] Opening connection to 34.47.176.25 on port 4901: Trying 34.47.176.25
[+] Opening connection to 34.47.176.25 on port 4901: Done
ciphertext: b'\x0e\xe8\x06\xea\xa9\xbfb\xa3\x0cL"\xdc\x9c\xe6-%\x0e\xe8\x06\xea\xa9\xbfb\xa3\x0cL"\xdc\x9c\xe6-%\xb8:\xda$\xc4\x1e\x87W\x17\x1ce\x1a8\xc1\x0f\r'
blocks    : [b'\x0e\xe8\x06\xea\xa9\xbfb\xa3\x0cL"\xdc\x9c\xe6-%', b'\x0e\xe8\x06\xea\xa9\xbfb\xa3\x0cL"\xdc\x9c\xe6-%', b'\xb8:\xda$\xc4\x1e\x87W\x17\x1ce\x1a8\xc1\x0f\r']
ebc, good...
[*] Closed connection to 34.47.176.25 port 4901


In [61]:
def get_ticket(username: bytes):
    io.recvuntil(b"> ")
    io.sendline(b"1")
    
    io.recvuntil(b"Username: ")
    io.sendline(username)
    
    io.recvuntil(b"Encrypted ticket:\n")
    ct = io.recvline().strip().decode()
    return ct

def usr_spoof(emperor: bytes):
    io.recvuntil(b"> ")
    io.sendline(b"2")

    io.recvuntil(b"Transmission: ")
    io.sendline(emperor)

    io.recvuntil(b"Ciphertext:\n")
    ct = io.recvline().strip().decode()
    return ct

def forged_ticket(ticket: bytes, emperor: bytes):
    payload = ticket + emperor
    
    io.recvuntil(b"> ")
    io.sendline(b"3")

    io.recvuntil(b"Encrypted ticket: ")
    io.sendline(payload.hex().encode())
    io.recvuntil(b"Access granted.\n")
    io.recvline()
    output = io.recvline()
    return output.strip().decode()

In [60]:
def find_flag(text: str):
    match = re.search(r"ACE\{.*?\}", text)
    return match.group(0) if match else None

In [ ]:
HOST = "34.47.176.25"
PORT = 4901
# context.log_level = 'debug'
context.log_level = 'error'

io = remote(HOST, PORT)
username = b"AAAAA"
payload = b"emperor"

clean_ticket = get_ticket(username)
print(f"size length: {len(clean_ticket)//2}")
print(clean_ticket)

spoof_ticket = clean_ticket[:16*4]
print(f"size length: {len(spoof_ticket)//2}")
print(spoof_ticket)


emperor = usr_spoof(payload)
print(f"\nsize length: {len(emperor)//2}")
print(emperor)

result = forged_ticket(bytes.fromhex(spoof_ticket),bytes.fromhex(emperor))

flag = find_flag(result)
print(flag)
io.close()

size length: 48
c68f51f43a443774d794c9b3b3c5829b2fd1326c6ce4dd30557a7eadd02cf2073a2754b5b47f4a5740c5880a95a0dd7d
size length: 32
c68f51f43a443774d794c9b3b3c5829b2fd1326c6ce4dd30557a7eadd02cf207

size length: 16
aa37dac75b67012b72debe8325bef782
<class 'str'>
ACE{Long_live_the_Empire_0a907347c304f052db464b9b1991b259}
